# 🚜 FarmifAI: Evaluación Integral de Modelos Conversacionales (SLM / GGUF)

Este notebook está diseñado para ejecutarse en **Google Colab** y realizar una evaluación rigurosa y exhaustiva de modelos de lenguaje pequeños (SLMs) basados en la arquitectura **Qwen3.5** (como `FarmifAI/FarmifAI_1.2_GGUF`), finetuneados para dar respuestas agrícolas en un formato estructurado de razonamiento (`<reasoning>`) y respuesta final (`<answer>`).

## Características Principales:
1. **Soporte Llama.cpp en GPU:** Inferencia local rápida con modelos cuantizados (`.gguf`).
2. **Descarga Dinámica desde Hugging Face:** Capacidad de listar y seleccionar cualquier cuantización del repositorio.
3. **Carga de Dataset JSONL:** Procesamiento de conversaciones con contexto agrícola (`<knowledge>`).
4. **10 Métricas de Evaluación:**
   - **Métricas Tradicionales/Lexicales:** Format Adherence, ROUGE (1, 2, L), BLEU, Índice de Legibilidad Flesch (Español), Similitud Coseno Semántica.
   - **Métricas Avanzadas / NLI / Consistencia:** Cross-Encoder NLI (Entailment/Contradiction/Neutral), SelfCheckGPT (Consistencia frente a muestreo estocástico).
   - **Métricas LLM as a Judge (DeepSeek / OpenRouter):** Faithfulness (Fidelidad contextual), Answer Relevancy (Relevancia a la pregunta), G-Eval (Coherencia, Consistencia, Fluidez).
5. **Resiliencia y Checkpointing:** Guardado progresivo en `eval_progress.jsonl` para prevenir pérdida de datos ante desconexiones.
6. **Reporte Visual Dashboard:** Tabulación en Pandas, Gráficos de Radar, Distribución de NLI y análisis de casos extremos.

## 1. Setup e Instalación de Dependencias

Instalamos las herramientas necesarias para la inferencia con `llama-cpp-python` y las librerías de evaluación NLP y comunicación con APIs de LLM Judges.

In [ ]:
%%capture
import os, sys, importlib.util

print("[INFO] Actualizando gestor de paquetes uv...")
!pip install --upgrade -qqq uv

print("[INFO] Instalando librerías de evaluación y utilidades...")
!uv pip install -qqq huggingface_hub rouge-score sacrebleu sentence-transformers scikit-learn openai pandas matplotlib seaborn tqdm nltk torch bert-score

import torch
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# --- INSTALACIÓN OPTIMIZADA DE LLAMA-CPP-PYTHON ---
# 1. Intentamos usar ruedas pre-compiladas (Pre-built Wheels) para CUDA en Colab (Tarda ~15 segundos)
print("[INFO] Instalando llama-cpp-python pre-compilado para GPU CUDA...")
cuda_ver = torch.version.cuda.replace(".", "")[:3] if torch.cuda.is_available() and torch.version.cuda else "124"
wheel_url = f"https://abetlen.github.io/llama-cpp-python/whl/cu{cuda_ver}"

res = os.system(f"uv pip install -qqq llama-cpp-python --extra-index-url {wheel_url}")

# 2. Si falla el wheel, compilamos limitando hilos y arquitectura para no saturar la RAM ni congelar Colab
if res != 0:
    print("[WARNING] Wheel no disponible. Compilando optimizado para la GPU de Colab (máx 2 min)...")
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=all-major"
    os.environ["MAX_JOBS"] = "4"
    !uv pip install --no-cache-dir -qqq llama-cpp-python bert-score

print("[OK] Instalación completada exitosamente.")

In [ ]:
import torch
import psutil

print("=" * 60)
print(" 🖥️ DIAGNÓSTICO DEL ENTORNO DE EVALUACIÓN")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detectada: {gpu_name}")
    print(f"📊 VRAM Disponible: {vram_total:.2f} GB")
    print(f"🚀 CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ ADVERTENCIA: No se detectó GPU CUDA. La inferencia será extremadamente lenta en CPU.")
    print("Por favor, ve a 'Entorno de ejecución' -> 'Cambiar tipo de entorno de ejecución' -> Selecciona 'T4 GPU'.")

ram_total = psutil.virtual_memory().total / (1024**3)
print(f"💾 RAM del Sistema: {ram_total:.2f} GB")
print("=" * 60)

## 2. Selección y Descarga Dinámica del Modelo GGUF

Nos conectamos a Hugging Face Hub para listar las diferentes cuantizaciones disponibles en el repositorio `FarmifAI/FarmifAI_1.2_GGUF`. Por defecto se evaluará `model-F16.gguf`, pero puedes elegir cuantizaciones más ligeras si lo deseas (ej. `Q4_K_M`, `Q8_0`).

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

HF_REPO_ID = "FarmifAI/FarmifAI_1.2_GGUF"
DEFAULT_GGUF_FILE = "model-F16.gguf"

api = HfApi()
try:
    print(
        f"[INFO] Listando modelos disponibles en el repositorio '{HF_REPO_ID}'...")
    repo_files = api.list_repo_files(repo_id=HF_REPO_ID)
    gguf_files = sorted([f for f in repo_files if f.endswith(".gguf")])

    print("\n📦 Cuantizaciones GGUF disponibles:")
    for idx, filename in enumerate(gguf_files):
        marker = " 👈 (Por defecto)" if filename == DEFAULT_GGUF_FILE else ""
        print(f"  [{idx}] {filename}{marker}")
except Exception as e:
    print(
        f"[WARNING] No se pudo obtener la lista remota ({e}). Usando lista por defecto.")
    gguf_files = [DEFAULT_GGUF_FILE]

# --- SELECCIÓN DEL MODELO ---
# Puedes cambiar el índice o el nombre del archivo si deseas evaluar otra cuantización
SELECTED_FILE = DEFAULT_GGUF_FILE

if SELECTED_FILE not in gguf_files and len(gguf_files) > 0:
    SELECTED_FILE = gguf_files[0]

print(f"\n[INFO] Descargando modelo seleccionado: {SELECTED_FILE}...")
model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=SELECTED_FILE)
print(f"✅ Modelo descargado y listo en ruta local: {model_path}")

## 3. Carga y Preparación del Dataset JSONL

Sube tu dataset de evaluación en formato JSONL. El notebook verificará que las conversaciones sigan la estructura estandarizada de FarmifAI:
- **System:** Instrucciones de formato (`<reasoning>...</reasoning>` y `<answer>...</answer>`).
- **User:** Contexto agrícola etiquetado dentro de `<knowledge>...</knowledge>` seguido de la pregunta del usuario.
- **Assistant:** Respuesta esperada (referencia) que contiene el razonamiento y la respuesta final.

In [ ]:
from google.colab import files
import json
import re

target_filename = "dataset_agricola_eval.jsonl"

# 1. Verificar si el archivo ya existe
if os.path.exists(target_filename):
    print(
        f"[INFO] El archivo '{target_filename}' ya existe localmente. Omitiendo la carga.")
    dataset_filename = target_filename
else:
    # 2. Si no existe, solicitar la carga
    print(f"[INFO] Selecciona y sube tu archivo '{target_filename}':")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError(
            "❌ No se subió ningún archivo. Por favor ejecuta la celda nuevamente y sube el archivo .jsonl.")

    dataset_filename = list(uploaded.keys())[0]
    print(f"✅ Archivo cargado: {dataset_filename}")

raw_data = []
with open(dataset_filename, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                raw_data.append(json.loads(line))
            except Exception as e:
                print(
                    f"[WARNING] Línea ignorada por error de formato JSON: {e}")

print(f"[INFO] Total de conversaciones encontradas: {len(raw_data)}")

# Función para extraer campos clave de cada muestra


def parse_convo_sample(sample, index):
    messages = sample.get("messages", [])
    system_msg = next((m["content"]
                      for m in messages if m["role"] == "system"), "")
    user_msg = next((m["content"]
                    for m in messages if m["role"] == "user"), "")
    assistant_msg = next((m["content"]
                         for m in messages if m["role"] == "assistant"), "")

    # Extraer knowledge y pregunta del usuario
    know_match = re.search(
        r'<knowledge>(.*?)</knowledge>', user_msg, re.DOTALL)
    knowledge = know_match.group(1).strip() if know_match else ""

    # La pregunta es el texto que queda fuera de <knowledge>...</knowledge>
    question = re.sub(r'<knowledge>.*?</knowledge>', '',
                      user_msg, flags=re.DOTALL).strip()
    if not question:
        question = user_msg.strip()

    # Extraer la respuesta final del asistente (sin el reasoning) para métricas de comparación directa
    ans_match = re.search(r'<answer>(.*?)</answer>', assistant_msg, re.DOTALL)
    ref_answer = ans_match.group(1).strip(
    ) if ans_match else assistant_msg.strip()

    return {
        "sample_index": index,
        "messages": messages,
        "system_msg": system_msg,
        "user_msg": user_msg,
        "knowledge": knowledge,
        "question": question,
        "reference_full": assistant_msg,
        "reference_answer": ref_answer
    }


eval_dataset = [parse_convo_sample(sample, idx)
                for idx, sample in enumerate(raw_data)]
print(
    f"✅ Dataset estructurado correctamente para evaluación ({len(eval_dataset)} muestras).")

# Mostrar ejemplo del dataset procesado
if eval_dataset:
    ej = eval_dataset[0]
    print("\n" + "-" * 50)
    print(f"🔍 EJEMPLO DE MUESTRA [Índice 0]:")
    print(f"📘 Pregunta: {ej['question'][:100]}...")
    print(f"📗 Contexto (<knowledge>): {ej['knowledge'][:100]}...")
    print(f"📙 Respuesta de Referencia: {ej['reference_answer'][:100]}...")
    print("-" * 50)

## 4. Configuración de Motores de Inferencia (SLM Local y LLM as a Judge)

En esta sección inicializamos:
1. **El Motor Local (Llama.cpp):** Carga el modelo GGUF descargado en la VRAM para generar las respuestas del asistente.
2. **El Cliente del Evaluador (LLM as a Judge):** Puedes usar **DeepSeek** u **OpenRouter** para evaluar métricas complejas de calidad (Faithfulness, Relevancy y G-Eval) utilizando salida estructurada en formato JSON.

In [ ]:
from llama_cpp import Llama
from openai import OpenAI

# --- 1. INICIALIZAR LLAMA.CPP LOCAL ---
print(f"[INFO] Cargando modelo local en Llama.cpp: {SELECTED_FILE}...")
llm_local = Llama(
    model_path=model_path,
    n_ctx=4096,          # Ventana de contexto para manejar RAG
    n_gpu_layers=-1,     # Cargar todas las capas posibles en GPU (-1)
    seed=42,
    verbose=False
)
print("✅ Motor Llama.cpp inicializado correctamente.")

# --- 2. CONFIGURACIÓN DEL LLM AS A JUDGE (API EXTERNA) ---
# Opciones de PROVEEDOR: "deepseek" o "openrouter"
JUDGE_PROVIDER = "deepseek"   # Cambiar a "openrouter" si prefieres usar OpenRouter

# 🔑 INGRESA AQUÍ TU API KEY:
# Deja vacío para que el notebook lo solicite de forma segura en ejecución
JUDGE_API_KEY = ""

# MODELO EVALUADOR:
# Para DeepSeek: "deepseek-v4-flash"
# Para OpenRouter: "nvidia/nemotron-3-ultra-550b-a55b:free"
JUDGE_MODEL = "deepseek-v4-flash" if JUDGE_PROVIDER == "deepseek" else "nvidia/nemotron-3-ultra-550b-a55b:free"

if not JUDGE_API_KEY:
    import getpass
    print(f"\n🔑 Por favor ingresa tu API Key para {JUDGE_PROVIDER.upper()}:")
    JUDGE_API_KEY = getpass.getpass()


def init_judge_client(provider, api_key):
    if provider.lower() == "deepseek":
        return OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
    elif provider.lower() == "openrouter":
        return OpenAI(
            api_key=api_key,
            base_url="https://openrouter.ai/api/v1",
            default_headers={
                "HTTP-Referer": "https://github.com/FarmifAI",
                "X-Title": "FarmifAI Model Evaluation"
            }
        )
    else:
        raise ValueError(f"Proveedor no soportado: {provider}")


judge_client = init_judge_client(JUDGE_PROVIDER, JUDGE_API_KEY)
print(
    f"✅ Cliente de LLM Judge configurado ({JUDGE_PROVIDER.upper()} - Modelo: {JUDGE_MODEL}).")

## 5. Implementación de las 10 Métricas de Evaluación

Aquí definimos el paquete completo de métricas que evaluarán cada respuesta generada:
1. **Format Adherence:** Verifica estrictamente la presencia y orden de `<reasoning>` y `<answer>`.
2. **ROUGE (1, 2, L):** Solapamiento léxico de secuencias y n-gramas.
3. **BLEU:** Precisión léxica basada en corpus.
4. **Índice de Legibilidad (Flesch Español):** Evalúa si el lenguaje es accesible para agricultores (>60 claro, <50 complejo).
5. **Similitud Coseno:** Similitud semántica con la respuesta de referencia usando embeddings multilingües.
   - **BERTScore (Precision, Recall, F1):** Evaluación de alineación semántica basada en representaciones de BERT en español.
6. **Cross-Encoder NLI:** Clasificación lógica de probabilidad entre *Entailment*, *Contradiction* y *Neutral* (Alucinación).
7. **SelfCheckGPT:** Consistencia semántica frente a 3 muestreos estocásticos a alta temperatura.
8. **Faithfulness (LLM Judge):** Evalúa si cada afirmación hecha en la respuesta está respaldada por el contexto `<knowledge>`.
9. **Answer Relevancy (LLM Judge):** Evalúa qué tan directa y completa es la respuesta con respecto a la pregunta.
10. **G-Eval (LLM Judge):** Análisis multicriterio CoT paso a paso evaluando *Coherencia*, *Consistencia* y *Fluidez*.

In [ ]:
import numpy as np
from rouge_score import rouge_scorer
import sacrebleu
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder
import bert_score
from bert_score import BERTScorer
import time
import nltk

print("[INFO] Cargando modelos de embedding y Cross-Encoder locales en GPU...")
embed_model = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2', device='cuda' if torch.cuda.is_available() else 'cpu')
nli_model = CrossEncoder('MoritzLaurer/mDeBERTa-v3-base-mnli-xnli',
                         device='cuda' if torch.cuda.is_available() else 'cpu')
rouge_evaluator = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
bert_scorer = BERTScorer(lang='es', device='cuda' if torch.cuda.is_available(
) else 'cpu', rescale_with_baseline=False)
print("✅ Modelos locales cargados exitosamente.")

# --- MÉTRICA 1: FORMAT ADHERENCE ---


def eval_format_adherence(text):
    reasoning_matches = re.findall(
        r'<reasoning>(.*?)</reasoning>', text, re.DOTALL)
    answer_matches = re.findall(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if len(reasoning_matches) == 1 and len(answer_matches) == 1:
        return 1.0 if text.find('<reasoning>') < text.find('<answer>') else 0.0
    return 0.0


def extract_generated_answer(text):
    match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()

# --- MÉTRICAS 2 & 3: ROUGE Y BLEU ---


def eval_rouge_bleu(ref_answer, gen_answer):
    if not gen_answer or not ref_answer:
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0, "bleu": 0.0}

    scores = rouge_evaluator.score(ref_answer, gen_answer)
    bleu = sacrebleu.sentence_bleu(gen_answer, [ref_answer]).score
    return {
        "rouge1": scores['rouge1'].fmeasure,
        "rouge2": scores['rouge2'].fmeasure,
        "rougeL": scores['rougeL'].fmeasure,
        "bleu": bleu
    }

# --- MÉTRICA 4: ÍNDICE DE LEGIBILIDAD (FLESCH ESPAÑOL - FERNÁNDEZ HUERTA) ---


def count_syllables_es(word):
    word = word.lower()
    vowels = 'aeiouáéíóúü'
    return max(1, sum(1 for char in word if char in vowels))


def eval_flesch_reading_ease_es(text):
    if not text:
        return 0.0
    sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
    words = re.findall(r'\w+', text)
    if not words or not sentences:
        return 0.0

    syllables = sum(count_syllables_es(w) for w in words)
    # Fórmula de Fernández Huerta validada para español
    score = 206.84 - 60.0 * (syllables / len(words)) - \
        1.02 * (len(words) / len(sentences))
    return round(max(0.0, min(100.0, score)), 2)

# --- MÉTRICA 5: SIMILITUD COSENO SEMÁNTICA ---


def eval_cosine_similarity(ref_answer, gen_answer):
    if not gen_answer or not ref_answer:
        return 0.0
    embs = embed_model.encode(
        [ref_answer, gen_answer], show_progress_bar=False)
    return float(cosine_similarity([embs[0]], [embs[1]])[0][0])


# --- MÉTRICA 5.1: BERTSCORE (PRECISION, RECALL, F1 SCORE) ---


def eval_bertscore(ref_answer, gen_answer):
    if not gen_answer or not ref_answer:
        return {"bertscore_precision": 0.0, "bertscore_recall": 0.0, "bertscore_f1": 0.0}
    P, R, F1 = bert_scorer.score([gen_answer], [ref_answer])
    return {
        "bertscore_precision": float(P[0].item()),
        "bertscore_recall": float(R[0].item()),
        "bertscore_f1": float(F1[0].item())
    }

# --- MÉTRICA 6: CROSS-ENCODER NLI (CLAIM-LEVEL MULTILINGÜE) ---


def eval_nli_relation(knowledge, gen_answer):
    if not knowledge or not gen_answer:
        return {"entailment": 0.0, "contradiction": 0.0, "neutral": 1.0, "label": "neutral"}

    # Extracción dinámica del mapeo id2label para aguantar cualquier modelo o cuantización
    id2label = getattr(nli_model.model.config, "id2label", {
                       0: "entailment", 1: "neutral", 2: "contradiction"})
    label_map = {str(v).lower(): int(k) for k, v in id2label.items()}

    # Descomponer en oraciones independientes (Claim/Sentence-level NLI)
    try:
        sentences = [s.strip() for s in nltk.tokenize.sent_tokenize(
            gen_answer, language="spanish") if len(s.strip()) > 10]
    except Exception:
        sentences = [s.strip() for s in re.split(
            r'[.!?]+', gen_answer) if len(s.strip()) > 10]
    if not sentences:
        sentences = [gen_answer.strip()]

    pairs = [(knowledge, s) for s in sentences]
    all_logits = nli_model.predict(pairs)
    if len(sentences) == 1 and all_logits.ndim == 1:
        all_logits = np.expand_dims(all_logits, axis=0)

    exp_logits = np.exp(all_logits - np.max(all_logits, axis=1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    ent_idx = label_map.get("entailment", 0)
    con_idx = label_map.get("contradiction", 2)
    neu_idx = label_map.get("neutral", 1)

    ent_probs = probs[:, ent_idx]
    con_probs = probs[:, con_idx]
    neu_probs = probs[:, neu_idx]

    max_con = float(np.max(con_probs))
    mean_ent = float(np.mean(ent_probs))
    max_ent = float(np.max(ent_probs))
    mean_neu = float(np.mean(neu_probs))

    # Lógica de clasificación factual para RAG agrícola
    if max_con > 0.5:
        top_label = "contradiction"
    elif mean_ent > 0.35 or max_ent > 0.65:
        top_label = "entailment"
    else:
        top_label = "neutral"

    return {
        "contradiction": max_con,
        "entailment": mean_ent,
        "neutral": mean_neu,
        "label": top_label
    }

# --- MÉTRICA 7: SELFCHECKGPT (CONSISTENCIA LOCAL) ---


def eval_selfcheck_gpt(prompt_chatml, main_answer, num_samples=3):
    if not main_answer:
        return 0.0
    stoch_answers = []

    for _ in range(num_samples):
        res = llm_local(
            prompt_chatml,
            max_tokens=384,
            stop=["<|im_end|>", "<|endoftext|>"],
            temperature=0.3,
            top_p=0.8
        )
        gen_text = res["choices"][0]["text"].strip()
        stoch_answers.append(extract_generated_answer(gen_text))
    main_emb = embed_model.encode([main_answer], show_progress_bar=False)
    stoch_embs = embed_model.encode(stoch_answers, show_progress_bar=False)

    sims = [cosine_similarity(main_emb, [s_emb])[0][0] for s_emb in stoch_embs]
    return float(np.mean(sims)) if sims else 0.0

# --- MÉTRICAS 8, 9 & 10: LLM AS A JUDGE (FAITHFULNESS, RELEVANCY, G-EVAL) ---


def call_llm_judge(system_prompt, user_prompt, max_retries=2):
    for attempt in range(max_retries):
        try:
            response = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,  # Baja temperatura para evaluación determinista y coherente
                response_format={"type": "json_object"}
            )
            content = response.choices[0].message.content
            return json.loads(content)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"[ERROR] Falló llamada a LLM Judge: {e}")
                return None
            time.sleep(2)


def eval_llm_judges(question, knowledge, gen_answer):
    if not gen_answer:
        return {"faithfulness": 0.0, "relevancy": 0.0, "g_eval": {"coherence": 0.0, "consistency": 0.0, "fluency": 0.0}}

    # 1. Faithfulness & Answer Relevancy (Evaluados en un llamado unificado para mayor eficiencia)
    prompt_sys_fr = (
        "Eres un evaluador experto en IA agrícola. Analiza la respuesta generada en base al contexto y la pregunta.\n"
        "Debes responder ESTRICTAMENTE en formato JSON con dos campos:\n"
        "1. 'faithfulness': puntaje de 0 a 5 indicando qué tanto de lo afirmado está respaldado por el contexto (5=100% respaldado, 0=contradicción total o invención).\n"
        "2. 'relevancy': puntaje de 0 a 5 indicando qué tan directa y completa es la respuesta a la pregunta del usuario (5=excelente y directa, 0=evasiva/irrelevante).\n"
        "Formato JSON esperado: {\"faithfulness\": <int 0-5>, \"relevancy\": <int 0-5>, \"reasoning\": \"<breve explicación>\"}"
    )
    prompt_usr_fr = f"Pregunta: {question}\n\nContexto (<knowledge>):\n{knowledge}\n\nRespuesta Generada:\n{gen_answer}"

    res_fr = call_llm_judge(prompt_sys_fr, prompt_usr_fr)
    faith_score = float(res_fr.get("faithfulness", 0.0)) if res_fr else 0.0
    rel_score = float(res_fr.get("relevancy", 0.0)) if res_fr else 0.0

    # 2. G-Eval (Coherence, Consistency, Fluency) con Cadena de Pensamiento (CoT)
    prompt_sys_geval = (
        "Eres un juez evaluador riguroso utilizando el framework G-Eval. Evalúa la respuesta en 3 dimensiones en escala de 1 a 5:\n"
        "1. 'coherence': Flujo lógico y organización estructural de la respuesta (1=desorganizada, 5=perfectamente estructurada).\n"
        "2. 'consistency': Ausencia de contradicciones lógicas internas o con los hechos agrícolas (1=contradictoria, 5=altamente consistente).\n"
        "3. 'fluency': Calidad gramatical en español, claridad y tono adaptado para un agricultor (1=incomprensible/robótico, 5=español natural, claro y excelente).\n"
        "Realiza un análisis paso a paso (CoT) y responde ESTRICTAMENTE en JSON con esta estructura:\n"
        "{\"coherence\": {\"score\": <int 1-5>, \"steps\": \"<análisis>\"}, \"consistency\": {\"score\": <int 1-5>, \"steps\": \"<análisis>\"}, \"fluency\": {\"score\": <int 1-5>, \"steps\": \"<análisis>\"}}"
    )

    res_geval = call_llm_judge(prompt_sys_geval, prompt_usr_fr)
    if res_geval:
        coh = float(res_geval.get("coherence", {}).get("score", 0.0))
        cons = float(res_geval.get("consistency", {}).get("score", 0.0))
        flue = float(res_geval.get("fluency", {}).get("score", 0.0))
    else:
        coh, cons, flue = 0.0, 0.0, 0.0

    return {
        "faithfulness": max(0.0, min(5.0, faith_score)),
        "relevancy": max(0.0, min(5.0, rel_score)),
        "g_eval": {
            "coherence": max(0.0, min(5.0, coh)),
            "consistency": max(0.0, min(5.0, cons)),
            "fluency": max(0.0, min(5.0, flue))
        }
    }

### ☁️ Respaldo en Google Drive
Ejecuta la siguiente celda para copiar el archivo de resultados `eval_progress.jsonl` directamente a Google Drive para tener un respaldo permanente.

In [ ]:
import shutil
from google.colab import drive

CHECKPOINT_FILE = "eval_progress.jsonl"

# Verificamos si el archivo de progreso existe
if not os.path.exists(CHECKPOINT_FILE):
    print(
        f"[INFO] El archivo '{CHECKPOINT_FILE}' no se encontró. Creando uno vacío...")
    # Crear el archivo vacío para evitar errores en shutil.copy
    with open(CHECKPOINT_FILE, 'w') as f:
        pass

try:
    print("[INFO] Conectando con Google Drive...")
    drive.mount('/content/drive')

    drive_folder = "/content/drive/MyDrive/FarmifAI_Evaluaciones"
    os.makedirs(drive_folder, exist_ok=True)

    dest_path = os.path.join(drive_folder, "eval_results.jsonl")
    shutil.copy(CHECKPOINT_FILE, dest_path)
    print(
        f"✅ ¡Respaldo guardado exitosamente en tu Google Drive!: {dest_path}")

except Exception as e:
    print(
        f"[INFO] No se montó Google Drive o se canceló el respaldo ({e}). El archivo sigue a salvo en el entorno local ('{CHECKPOINT_FILE}').")

## 6. Bucle de Evaluación con Sistema de Checkpointing y Resiliencia

Para proteger tu progreso ante posibles desconexiones o interrupciones de Colab, implementamos un sistema de **Checkpointing Automático**:
1. Cada muestra evaluada se guarda inmediatamente en un archivo JSONL en el disco local (`eval_progress.jsonl`).
2. Si la ejecución se interrumpe, **simplemente vuelve a ejecutar esta celda**: el notebook leerá los IDs ya completados y continuará exactamente donde se quedó.
3. Al final, te damos la opción de guardar una copia de seguridad en tu Google Drive.

In [ ]:
from tqdm.notebook import tqdm


# 1. Cargar progreso existente si hay una sesión previa interrumpida
completed_indices = set()
results_list = []

if os.path.exists(CHECKPOINT_FILE):
    print(
        f"[INFO] Archivo de checkpoint encontrado: '{CHECKPOINT_FILE}'. Leyendo progreso previo...")
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    res_obj = json.loads(line)
                    completed_indices.add(res_obj["sample_index"])
                    results_list.append(res_obj)
                except Exception as e:
                    pass
    print(
        f"🔄 Progreso recuperado: {len(completed_indices)} muestras ya evaluadas.")
else:
    print("[INFO] No se encontró checkpoint previo. Iniciando evaluación desde cero.")

# 2. Configurar el tamaño de muestra a evaluar (puedes ajustar este número según tu tiempo y presupuesto)
# Cambia por ej. a 20 o 50 si quieres una prueba rápida
MAX_SAMPLES_TO_EVALUATE = len(eval_dataset)

# 3. Bucle de Evaluación
print(
    f"\n🚀 INICIANDO EVALUACIÓN (Total objetivo: {min(MAX_SAMPLES_TO_EVALUATE, len(eval_dataset))} muestras)...")

for sample in tqdm(eval_dataset[:MAX_SAMPLES_TO_EVALUATE], desc="Evaluando Modelo SLM"):
    idx = sample["sample_index"]

    # Si ya se evaluó previamente, saltar sin gastar recursos ni llamadas API
    if idx in completed_indices:
        continue

    # --- PASO A: Formatear Prompt en ChatML para el Modelo ---
    # Garantizamos que el formato sea idéntico al entrenamiento de Qwen3.5
    prompt_chatml = f"<|im_start|>system\n{sample['system_msg']}<|im_end|>\n"
    prompt_chatml += f"<|im_start|>user\n{sample['user_msg']}<|im_end|>\n<|im_start|>assistant\n"

    # --- PASO B: Generar Respuesta del Modelo en Llama.cpp ---
    gen_res = llm_local(
        prompt_chatml,
        max_tokens=512,
        stop=["<|im_end|>", "<|endoftext|>"],
        temperature=0.3,
        top_p=0.8
    )
    generated_full = gen_res["choices"][0]["text"].strip()
    generated_answer = extract_generated_answer(generated_full)

    # --- PASO C: Calcular las 10 Métricas ---
    fmt_score = eval_format_adherence(generated_full)
    rb_scores = eval_rouge_bleu(sample["reference_answer"], generated_answer)
    flesch_score = eval_flesch_reading_ease_es(generated_answer)
    cos_score = eval_cosine_similarity(
        sample["reference_answer"], generated_answer)
    bert_scores = eval_bertscore(
        sample["reference_answer"], generated_answer)
    nli_res = eval_nli_relation(sample["knowledge"], generated_answer)
    selfcheck_score = eval_selfcheck_gpt(
        prompt_chatml, generated_answer, num_samples=3)
    judge_res = eval_llm_judges(
        sample["question"], sample["knowledge"], generated_answer)

    # --- PASO D: Consolidar Registro ---
    eval_record = {
        "sample_index": idx,
        "question": sample["question"],
        "knowledge": sample["knowledge"],
        "reference_answer": sample["reference_answer"],
        "generated_full": generated_full,
        "generated_answer": generated_answer,
        "metrics": {
            "format_adherence": fmt_score,
            "rouge1": rb_scores["rouge1"],
            "rouge2": rb_scores["rouge2"],
            "rougeL": rb_scores["rougeL"],
            "bleu": rb_scores["bleu"],
            "flesch_reading_ease": flesch_score,
            "cosine_similarity": cos_score,
            "bertscore_precision": bert_scores["bertscore_precision"],
            "bertscore_recall": bert_scores["bertscore_recall"],
            "bertscore_f1": bert_scores["bertscore_f1"],
            "nli_entailment": nli_res["entailment"],
            "nli_contradiction": nli_res["contradiction"],
            "nli_neutral": nli_res["neutral"],
            "nli_label": nli_res["label"],
            "selfcheck_consistency": selfcheck_score,
            "faithfulness": judge_res["faithfulness"],
            "answer_relevancy": judge_res["relevancy"],
            "geval_coherence": judge_res["g_eval"]["coherence"],
            "geval_consistency": judge_res["g_eval"]["consistency"],
            "geval_fluency": judge_res["g_eval"]["fluency"]
        }
    }

    # --- PASO E: Guardar en Checkpoint Progresivamente ---
    with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(eval_record, ensure_ascii=False) + "\n")

    shutil.copy(CHECKPOINT_FILE, dest_path)

    completed_indices.add(idx)
    results_list.append(eval_record)

print(
    f"\n✅ ¡EVALUACIÓN COMPLETADA CON ÉXITO! Total evaluado: {len(results_list)} muestras.")

## 7. Reporte Ejecutivo y Visualizaciones de Calidad

En esta sección final generamos un resumen detallado y fácil de interpretar para el equipo de FarmifAI:
1. **Tabla Resumen Estadístico:** Promedios, desviaciones estándar y semáforo de calidad.
   - Incluye submétricas de BERTScore (Precision, Recall, F1).
2. **Gráfico de Radar (Spider Chart):** Vista holística del rendimiento del modelo normalizado a escala de 0 a 100%.
3. **Distribución NLI (Cross-Encoder):** Porcentaje de respuestas respaldadas (*Entailment*) vs Alucinaciones (*Neutral/Contradiction*).
4. **Análisis de Casos Clínicos:** Exploración de las mejores y peores respuestas generadas para auditoría técnica.
5. **Excepción Legibilidad Flesch (ES):** El rango ideal para el contexto agrícola colombiano es 55.0 - 75.0 (equilibrado).


### (Opcional) Subir archivo de resultados

Ejecutar en caso de que ya se tengan los resultados y solo se requiera el reporte.

In [ ]:
import json
import os
from google.colab import files

print("Sube tu archivo de resultados (.jsonl) para generar el reporte de evaluación:")
uploaded = files.upload()

# Inicializamos las variables que necesita el bloque de visualización
results_list = []
SELECTED_FILE = ""

if uploaded:
    # Tomar el nombre del archivo que el usuario acaba de subir
    SELECTED_FILE = list(uploaded.keys())[0]
    print(
        f"\n[INFO] Archivo '{SELECTED_FILE}' cargado en el entorno. Leyendo datos...")

    # Leer el archivo JSONL línea por línea
    with open(SELECTED_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    res_obj = json.loads(line)
                    results_list.append(res_obj)
                except json.JSONDecodeError as e:
                    print(
                        f"[WARNING] Error al decodificar una línea, se omitirá. Detalles: {e}")

    print(
        f"✅ ¡Datos cargados con éxito! Total de muestras recuperadas: {len(results_list)}")
    print("➡️ Ya puedes ejecutar el bloque del dashboard (el que contiene matplotlib y pandas).")
else:
    print("❌ No se subió ningún archivo. Por favor, intenta de nuevo.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from IPython.display import display

# ====================================================
# 1. PREPARAR DATOS
# ====================================================
df_res = pd.json_normalize([r["metrics"] for r in results_list])
n_samples = len(df_res)

# Flags binarios derivados de la ETIQUETA NLI
df_res["nli_entailment_flag"] = (df_res["nli_label"] == "entailment").astype(
    int) if "nli_label" in df_res.columns else 0

# ====================================================
# 2. TABLA PRINCIPAL: SOLO MÉTRICAS CONTINUAS
# ====================================================
METRIC_DEFS = [
    ("1. Format Adherence (%)",         "format_adherence",
     100, [95, 80, 60],       False),
    ("2. ROUGE-1 (F1)",                 "rouge1",
     1, [0.50, 0.30, 0.15], False),
    ("   ROUGE-2 (F1)",                 "rouge2",
     1, [0.30, 0.15, 0.05], False),
    ("   ROUGE-L (F1)",                 "rougeL",
     1, [0.50, 0.30, 0.15], False),
    ("3. BLEU Score",                   "bleu",
     1, [30, 15, 5],        False),
    ("4. Legibilidad Flesch (ES)",
     "flesch_reading_ease",    1, [80, 60, 40],       False),
    ("5. Similitud Coseno Semántica",
     "cosine_similarity",      1, [0.85, 0.70, 0.50], False),
    ("   BERTScore Precision",          "bertscore_precision",
     1, [0.85, 0.70, 0.50], False),
    ("   BERTScore Recall",             "bertscore_recall",
     1, [0.85, 0.70, 0.50], False),
    ("   BERTScore F1 Score",           "bertscore_f1",
     1, [0.85, 0.70, 0.50], False),
    ("6. SelfCheckGPT Consistencia",
     "selfcheck_consistency",  1, [0.85, 0.70, 0.50], False),
    ("7. Faithfulness (Judge 0-5)",     "faithfulness",
     1, [4.5, 3.5, 2.5],    False),
    ("8. Answer Relevancy (Judge 0-5)",
     "answer_relevancy",       1, [4.5, 3.5, 2.5],    False),
    ("9. G-Eval Coherencia (0-5)",      "geval_coherence",
     1, [4.5, 3.5, 2.5],    False),
    ("   G-Eval Consistencia (0-5)",
     "geval_consistency",      1, [4.5, 3.5, 2.5],    False),
    ("   G-Eval Fluidez (0-5)",         "geval_fluency",
     1, [4.5, 3.5, 2.5],    False),
]

TABLE_COLORS = {"Excelente": "#c6efce", "Bueno": "#ffeb9c",
                "Regular": "#ffd8b1", "Bajo": "#ffc7ce"}
VIVID_COLORS = {"Excelente": "#28a745", "Bueno": "#8bc34a",
                "Regular": "#fd7e14", "Bajo": "#dc3545"}


def calificar(col, valor, umbrales, invertido):
    if col == "flesch_reading_ease":
        # Excepción dominio agrícola colombiano: Rango equilibrado (55-75 es Excelente)
        if 55.0 <= valor <= 75.0:
            return "Excelente"
        elif (50.0 <= valor < 55.0) or (75.0 < valor <= 85.0):
            return "Bueno"
        elif (40.0 <= valor < 50.0) or (85.0 < valor <= 90.0):
            return "Regular"
        else:
            return "Bajo"
    a, b, c = umbrales
    if invertido:
        if valor <= a:
            return "Excelente"
        if valor <= b:
            return "Bueno"
        if valor <= c:
            return "Regular"
        return "Bajo"
    else:
        if valor >= a:
            return "Excelente"
        if valor >= b:
            return "Bueno"
        if valor >= c:
            return "Regular"
        return "Bajo"


rows = []
for nombre, col, factor, umbrales, invertido in METRIC_DEFS:
    if col in df_res.columns:
        promedio = df_res[col].mean() * factor
        std = df_res[col].std() * factor
    else:
        promedio = 0.0
        std = 0.0
    rows.append({
        "Métrica": nombre,
        "Promedio": promedio,
        "Desv. Estándar": std,
        "Evaluación": calificar(col, promedio, umbrales, invertido)
    })
summary_stats = pd.DataFrame(rows)


def resaltar_fila(row):
    color = TABLE_COLORS.get(row["Evaluación"], "#ffffff")
    return [f"background-color: {color}; color: #1a1a1a"] * len(row)


def emoji_evaluacion(v):
    return {"Excelente": "🟢 Excelente", "Bueno": "🟡 Bueno",
            "Regular": "🟠 Regular", "Bajo": "🔴 Bajo"}.get(v, v)


summary_display = summary_stats.copy()
summary_display["Evaluación"] = summary_display["Evaluación"].map(
    emoji_evaluacion)

styled = (
    summary_display.style
    .apply(lambda row: resaltar_fila(summary_stats.loc[row.name]), axis=1)
    .format({"Promedio": "{:,.3f}", "Desv. Estándar": "{:,.3f}"})
    .set_properties(**{"text-align": "left", "font-size": "13px", "color": "#1a1a1a"})
    .set_table_styles([{"selector": "th", "props": [
        ("text-align", "left"), ("background-color", "#2b5c8f"),
        ("color", "white"), ("font-size", "13px")]}])
    .hide(axis="index")
)

print("=" * 90)
print(
    f" 📊 REPORTE DE EVALUACIÓN FINAL — MODELO: {SELECTED_FILE}   |   Muestras: {n_samples}")
print("=" * 90)
display(styled)
print("Leyenda: 🟢 Excelente   🟡 Bueno   🟠 Regular   🔴 Bajo")
print("Nota 1: Faithfulness/G-Eval son puntajes acotados 0-5; una Desv. Estándar alta")
print("con la media cerca del techo/piso puede exceder nominalmente el rango — es")
print("normal, refleja desacuerdo entre jueces, no un error de cómputo.")
print("Nota 2: Para la Legibilidad Flesch (ES) en agricultores colombianos, el rango idóneo es")
print("moderado (55.0 - 75.0) para asegurar claridad campesina sin perder precisión técnica.\n")

# ====================================================
# 3. TABLA SEPARADA: CLASIFICACIÓN NLI
# ====================================================


def wilson_ci(count, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    phat = count / n
    denom = 1 + z**2 / n
    centro = (phat + z**2 / (2 * n)) / denom
    margen = (z * np.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n)) / denom
    lo = max(0.0, centro - margen) * 100
    hi = min(1.0, centro + margen) * 100
    return lo, hi


NLI_UMBRAL = {
    "Entailment (Soportado)":    ([70, 50, 30], False),
    "Neutral (Alucinación)":     ([10, 25, 40], True),
    "Contradicción":             ([5, 15, 30],  True),
}

nli_rows = []
for label, disp in [("entailment", "Entailment (Soportado)"),
                    ("neutral", "Neutral (Alucinación)"),
                    ("contradiction", "Contradicción")]:
    count = int((df_res["nli_label"] == label).sum()
                ) if "nli_label" in df_res.columns else 0
    pct = (count / n_samples * 100) if n_samples > 0 else 0.0
    lo, hi = wilson_ci(count, n_samples)
    umbrales, invertido = NLI_UMBRAL[disp]
    nli_rows.append({
        "Categoría NLI": disp,
        "Conteo": count,
        "% del Total": pct,
        "IC 95% (Wilson)": f"{lo:.1f}% – {hi:.1f}%",
        "Evaluación": calificar("nli", pct, umbrales, invertido)
    })
nli_table = pd.DataFrame(nli_rows)


def resaltar_nli(row):
    color = TABLE_COLORS.get(row["Evaluación"], "#ffffff")
    return [f"background-color: {color}; color: #1a1a1a"] * len(row)


nli_display = nli_table.copy()
nli_display["Evaluación"] = nli_display["Evaluación"].map(emoji_evaluacion)

styled_nli = (
    nli_display.style
    .apply(lambda row: resaltar_nli(nli_table.loc[row.name]), axis=1)
    .format({"% del Total": "{:.1f}%"})
    .set_properties(**{"text-align": "left", "font-size": "13px", "color": "#1a1a1a"})
    .set_table_styles([{"selector": "th", "props": [
        ("text-align", "left"), ("background-color", "#2b5c8f"),
        ("color", "white"), ("font-size", "13px")]}])
    .hide(axis="index")
)

print("=" * 90)
print(" 🛡️  CLASIFICACIÓN NLI (distribución de la etiqueta final por muestra)")
print("=" * 90)
display(styled_nli)
print("El IC 95% (Wilson) indica el rango plausible del % real dado el tamaño de")
print(f"la muestra (n={n_samples}); se estrecha con más datos evaluados.\n")

# ====================================================
# 4. DASHBOARD DE GRÁFICOS (2x2)
# ====================================================
plt.rcParams.update({"font.size": 10})
fig = plt.figure(figsize=(20, 17))
gs = fig.add_gridspec(2, 2, hspace=0.55, wspace=0.30,
                      top=0.90, bottom=0.05, left=0.06, right=0.97)

ax_radar = fig.add_subplot(gs[0, 0], projection="polar")
ax_nli = fig.add_subplot(gs[0, 1])
ax_judge = fig.add_subplot(gs[1, 0])
ax_box = fig.add_subplot(gs[1, 1])

# --- A. RADAR: rendimiento holístico normalizado 0-100% ---
radar_labels = [
    "Format\nAdherence", "ROUGE-L", "Similitud\nSemántica", "BERTScore\nF1",
    "NLI\nEntailment", "SelfCheck\nConsistencia", "Faithfulness\n(Judge)",
    "Relevancia\n(Judge)", "G-Eval\nCoherencia", "G-Eval\nFluidez"
]
radar_raw = [
    df_res["format_adherence"].mean(
    ) * 100 if "format_adherence" in df_res else 0.0,
    df_res["rougeL"].mean() * 100 if "rougeL" in df_res else 0.0,
    df_res["cosine_similarity"].mean(
    ) * 100 if "cosine_similarity" in df_res else 0.0,
    (df_res["bertscore_f1"].mean() * 100) if "bertscore_f1" in df_res else 0.0,
    df_res["nli_entailment_flag"].mean(
    ) * 100 if "nli_entailment_flag" in df_res else 0.0,
    df_res["selfcheck_consistency"].mean(
    ) * 100 if "selfcheck_consistency" in df_res else 0.0,
    (df_res["faithfulness"].mean() / 5.0) *
    100 if "faithfulness" in df_res else 0.0,
    (df_res["answer_relevancy"].mean() / 5.0) *
    100 if "answer_relevancy" in df_res else 0.0,
    (df_res["geval_coherence"].mean() / 5.0) *
    100 if "geval_coherence" in df_res else 0.0,
    (df_res["geval_fluency"].mean() / 5.0) *
    100 if "geval_fluency" in df_res else 0.0,
]
radar_values = radar_raw + radar_raw[:1]
angles = np.linspace(0, 2 * np.pi, len(radar_labels), endpoint=False).tolist()
angles += angles[:1]

ax_radar.plot(angles, radar_values, color="#2b5c8f", linewidth=2.5)
ax_radar.fill(angles, radar_values, color="#2b5c8f", alpha=0.25)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(radar_labels, size=9)
ax_radar.tick_params(axis="x", pad=22)
ax_radar.set_ylim(0, 100)
ax_radar.set_yticks([25, 50, 75, 100])
ax_radar.set_yticklabels(["25", "50", "75", "100"], size=8, color="gray")
ax_radar.set_title("Rendimiento Holístico (0-100%)",
                   size=13, weight="bold", pad=45)

# --- B. DISTRIBUCIÓN NLI (% de respuestas) ---
nli_order = ["entailment", "neutral", "contradiction"]
nli_counts = df_res["nli_label"].value_counts().reindex(nli_order).fillna(
    0) if "nli_label" in df_res.columns else pd.Series([0, 0, 0], index=nli_order)
nli_pcts = (nli_counts / max(1, n_samples)) * 100
bar_colors = ["#28a745", "#ffc107", "#dc3545"]
labels_nli = ["Entailment\n(Soportado)", "Neutral\n(No verificable)",
              "Contradicción\n(Alucinación)"]

bars = ax_nli.bar(labels_nli, nli_pcts.values,
                  color=bar_colors, edgecolor="black", alpha=0.85)
ax_nli.set_title("Distribución NLI — Fidelidad Fáctica",
                 size=13, weight="bold")
ax_nli.set_ylabel("Porcentaje de Respuestas (%)")
ax_nli.set_ylim(0, 100)
for bar, v in zip(bars, nli_pcts.values):
    ax_nli.text(bar.get_x() + bar.get_width() / 2, v + 2.0,
                f"{v:.1f}%", ha="center", va="bottom", weight="bold", size=10)

# --- C. PUNTAJES LLM-AS-A-JUDGE (0-5) ---
judge_cols = ["faithfulness", "answer_relevancy",
              "geval_coherence", "geval_consistency", "geval_fluency"]
judge_labels = ["Faithfulness", "Answer\nRelevancy",
                "G-Eval\nCoherencia", "G-Eval\nConsistencia", "G-Eval\nFluidez"]
judge_means = np.array(
    [df_res[c].mean() if c in df_res else 0.0 for c in judge_cols])
judge_stds = np.array(
    [df_res[c].std() if c in df_res else 0.0 for c in judge_cols])

lower_err = np.minimum(judge_stds, judge_means - 0)
upper_err = np.minimum(judge_stds, 5 - judge_means)
xerr = np.vstack([lower_err, upper_err])


def color_judge(v):
    if v >= 4.5:
        return VIVID_COLORS["Excelente"]
    if v >= 3.5:
        return VIVID_COLORS["Bueno"]
    if v >= 2.5:
        return VIVID_COLORS["Regular"]
    return VIVID_COLORS["Bajo"]


y_pos = np.arange(len(judge_labels))
ax_judge.barh(y_pos, judge_means, xerr=xerr, color=[color_judge(v) for v in judge_means],
              edgecolor="black", alpha=0.9, capsize=4, error_kw={"elinewidth": 1.3})
ax_judge.set_yticks(y_pos)
ax_judge.set_yticklabels(judge_labels)
ax_judge.set_xlim(0, 5)
ax_judge.set_xlabel("Puntaje (0-5) — barra de error recortada al rango válido")
ax_judge.set_title(
    "Evaluación LLM-as-a-Judge (± Desv. Est., recortada)", size=13, weight="bold")
ax_judge.invert_yaxis()
for i, m in enumerate(judge_means):
    ax_judge.text(m / 2, i, f"{m:.2f}", ha="center", va="center", weight="bold",
                  size=10, color="black",
                  path_effects=[pe.withStroke(linewidth=3, foreground="white")])

# --- D. DISTRIBUCIÓN POR MUESTRA (variabilidad, 0-1) ---
dist_cols = ["rouge1", "rougeL", "cosine_similarity", "selfcheck_consistency"]
dist_labels = ["ROUGE-1", "ROUGE-L",
               "Similitud\nCoseno", "SelfCheck\nConsistencia"]
data_to_plot = [df_res[c].dropna().values if c in df_res else np.array([])
                for c in dist_cols]

bp = ax_box.boxplot(data_to_plot, tick_labels=dist_labels,
                    patch_artist=True, showmeans=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#a8c8e8")
ax_box.set_title("Distribución de Métricas por Muestra",
                 size=13, weight="bold")
ax_box.set_ylabel("Valor (0-1)")
ax_box.set_ylim(0, 1.05)
ax_box.grid(axis="y", alpha=0.3)

fig.suptitle(
    f"Dashboard de Evaluación — {SELECTED_FILE}", size=16, weight="bold", y=0.985)
plt.show()

In [ ]:
print("=" * 75)
print(" 🔬 ANÁLISIS CUALITATIVO: CASOS EXTREMOS (BEST & WORST CASES)")
print("=" * 75)

# Calcular un score compuesto de calidad global (0 a 100)
df_res["global_score"] = (
    (df_res["format_adherence"] * 20) +
    (df_res["cosine_similarity"] * 20) +
    (df_res["nli_entailment"] * 20) +
    ((df_res["faithfulness"] / 5.0) * 20) +
    ((df_res["answer_relevancy"] / 5.0) * 20)
)

best_idx = df_res["global_score"].idxmax()
worst_idx = df_res["global_score"].idxmin()

best_case = results_list[best_idx]
worst_case = results_list[worst_idx]


def print_case_card(case, title, color_emoji):
    print(f"\n{color_emoji} {title} [Muestra #{case['sample_index']}] — Score Global: {df_res.loc[df_res['global_score'].idxmax() if 'MEJOR' in title else df_res['global_score'].idxmin(), 'global_score']:.1f}/100")
    print(f"📌 PREGUNTA: {case['question']}")
    print(f"📖 CONTEXTO (<knowledge>): {case['knowledge'][:200]}...")
    print(f"✨ RESPUESTA GENERADA: {case['generated_answer']}")
    print(f"📊 MÉTRICAS CLAVE: Faithfulness={case['metrics']['faithfulness']}/5 | Relevancia={case['metrics']['answer_relevancy']}/5 | NLI={case['metrics']['nli_label'].upper()} | Legibilidad Flesch={case['metrics']['flesch_reading_ease']}")
    print("-" * 75)


print_case_card(
    best_case, "MEJOR RESPUESTA GENERADA (Alta Fidelidad y Relevancia)", "🏆")
print_case_card(
    worst_case, "PEOR RESPUESTA GENERADA (Área de Oportunidad / Alucinación)", "⚠️")